# Subtitles Showcase

This notebook demonstrates how to extract and process text from `.srt` subtitle files, clean the text, remove duplicates, and finally extract vocabulary tokens (verbs and other words) using the Natural Language API.

In [1]:
%load_ext autoreload

In [3]:
%autoreload 2
from setup_imports import *  # noqa: F401,F403


from subtitles import read_subtitles, process_subtitles, get_subtitle_tokens
from nlp import get_text_tokens, get_verbs_and_vocab
from phrases.phrase_model import get_unique_tokens_from_phrases
from phrases.search import get_phrases_by_collection, find_phrases_by_vocab_dict

## 1. Reading Subtitles
We use `pysrt` to read the subtitles from the `data/subtitles/` directory.

In [4]:
subtitle_file = "../data/subtitles/Trouble_sv_only.srt"

try:
    subs = read_subtitles(subtitle_file)
    print(f"Successfully loaded {len(subs)} subtitle entries.")
    print("\nFirst 3 entries:")
    for i in range(min(3, len(subs))):
        print(f"{i+1}: {subs[i].text}")
except Exception as e:
    print(f"Error reading subtitles: {e}")

Successfully loaded 1543 subtitle entries.

First 3 entries:
1: [visslar en melodi]
2: Men snälla, tyst.
3: Du väntar här. När jag messar dig
kommer du in med resten av grejerna.


## 2. Processing Subtitles
We will now clean the subtitles by removing Closed Captioning (CC) info enclosed in square brackets `[like this]`, replace newlines with spaces, strip extra whitespace, and remove duplicates to get a unique list of phrases.

In [16]:
import string
for char in ("-Vi"):
    print(char, char in string.punctuation) 

- True
V False
i False


In [19]:
unique_phrases = process_subtitles(subs, language_code="sv", split_on_space=True)

print(f"Total unique phrases extracted: {len(unique_phrases)}")
print("\nSample of 10 unique phrases:")
for phrase in unique_phrases[:100]:
    print(f"{phrase}")

Total unique phrases extracted: 1313

Sample of 10 unique phrases:
Men snälla tyst
Du väntar här När jag messar dig kommer du in med resten av grejerna
Ja Capite
Ska vi sticka
Vad är det här för nåt
Det är ett kilo det här Vi sa 30  Mm
Vi har dem Men pengarna först
Vi står utanför Bamboo Bamboo
Det är bekräftat de misstänkta befinner sig i lokalen
Kör
Enhet A vänster Enhet B bakvägen
Ja Uppfattat
Portkoden
1490
Vänster Vänster Vänster
Ja grabbar det här kan ni In och plocka dem
så tar vi en AW på Stopet sen
Då kallar jag in grejerna
Vapen Vapen
Vad händer Vad fan händer
Nej
Angripare avväpnad Vi avancerar
Polis Polis Visa händerna
Ner på golvet Släpp vapnen Den tog i västen Det är lugnt
Nu tar vi dem
Ner på golvet Ner på golvet
Händerna över huvudet Håll käften
Var fan är alla droger
Det saknas en massa Var fan är resten
Hej
Hej hej ta det lugnt Okej okej
Chilla okej okej Vänta
Här
Här Ta grejerna Skjut inte
Åh shit
Mina damer och herrar detta är kapten Conny Rundkvist från cockpit
Som

## 3. Extracting Tokens
Using `src.nlp`, we'll run a sample of these phrases through the Google Cloud Natural Language API to extract lemmas and group them into `verbs` and `vocab`.

In [20]:
# To avoid long API wait times in this showcase, we'll process just the first 20 phrases.
sample_phrases = unique_phrases[:20]

print(f"Extracting vocabulary from {len(unique_phrases)} phrases...")
vocab_dict = get_verbs_and_vocab(unique_phrases, language="sv-se")

print("\n--- Extracted Verbs ---")
print(vocab_dict.get('verbs', []))

print("\n--- Extracted Vocab (Non-Verbs) ---")
print(vocab_dict.get('vocab', []))

Extracting vocabulary from 1313 phrases...

--- Extracted Verbs ---
['bryta', 'vädja', 'skynda', 'öppna', 'häng', 'ändra', 'titta', 'bestäma', 'beställ', 'lämna', 'gilla', 'svara', 'precis', 'skylla', 'ladda', 'setts', 'förstå', '…', 'rikta', 'mena', 'fick', 'ha', 'landa', 'finnas', 'döma', 'börja', 'låta', 'är', 'sticka', 'går', 'möta', 'kalla', 'vinna', 'känna', 'tro', 'pulla', 'skit', 'spela', 'hjälpa', 'avancera', 'snacka', 'ring', 'vilja', 'jävlar', 'spänna', 'bruka', 'strömma', 'fatta', 'vara', 'klara', 'druckit', 'påträffa', 'följ', 'parkoppla', 'raka', 'filma', 'hoppa', 'saa', 'pilla', 'stoppa', 'skottskada', 'inträffa', 'skola', 'undra', 'råka', 'fan', 'plocka', 'snöar', 'förändra', 'tracka', 'rapportera', 'fronta', 'starta', 'sakna', 'prata', 'flög', 'gratta', 'ropa', 'glömma', 'flyga', 'grepa', 'berätta', 'fixa', 'kolla', 'heta', 'okej', 'läsa', 'hända', 'syssla', 'falla', 'sjunga', 'vårdades', 'befinna', 'uppfattad', 'hej', 'köra', 'åka', 'störta', 'sova', 'försvann', 'påko

In [21]:
p, m = find_phrases_by_vocab_dict(vocab_dict, language="sv-SE")

In [27]:
from subtitles import filter_wiktionary_words
missing_verbs = m['verbs']
missing_valid_verbs = filter_wiktionary_words(missing_verbs, "sv")
missing_vocab = m['vocab']
missing_valid_vocab = filter_wiktionary_words(missing_vocab, "sv")

In [29]:
vocab_dict_save = {"verbs" : missing_valid_verbs, "vocab" : missing_valid_vocab}

In [30]:
from utils import save_json
save_json(vocab_dict_save, file_path="../data/total_vocab_dict.json")

In [ ]:
[phrase.translations["sv-SE"].text for phrase in p]

['Jag vet inte om sajten är öppen för alla',
 'Ingenting kommer att spela någon roll då.',
 'Hon undrade vart han hade tagit vägen',
 'Vi ska titta på en film ikväll',
 'Det är snälla av dig att tänka så',
 'De höll fast vid planen',
 'Ett litet barn med en kvinna',
 'Jag skriver ut dokument alldeles för mycket',
 'Jag känner igen det här stället',
 'Min bror har sin beskärda del',
 'inget annat än sanningen',
 'Kom ihåg att ringa henne',
 'Hon gick fram till byggnaden',
 'Hon väntade i två timmar',
 'Kan du ställa dig där borta?',
 'Är du redo att beställa?',
 'Stäng dörren, tack',
 'Utan tvekan det bästa resultatet',
 'Gör klart dina läxor nu',
 'Hennes kontor ligger längre bort',
 'Hur säger man det?',
 'Jag laddar min telefon varje dag',
 'Jag slutar jobba tidigt',
 'Jag blev trött snabbt',
 'Jag antar väl att du har rätt',
 'Jag har blandade känslor kring detta',
 'Jag brukar äta frukost',
 'Jag letar efter en särskild maskin',
 'Myggor dödar många människor',
 'Min fars gamla klo

In [ ]:
film_tokens = get_subtitle_tokens(unique_phrases, "sv")

In [ ]:
sorted(film_tokens)

In [ ]:
lm1000 = get_phrases_by_collection("LM1000")

In [ ]:
lm1000_str = [(x._get_translation('sv-SE').text) for x in lm1000[:10]]

In [13]:
results = find_phrases_by_vocab_dict(
vocab_dict={"verbs": ["springa", "äta"], "vocab": ["hund", "bil"]},
language="sv-SE",
collection="LM1000")
for p in results:
    print(p.english, p.translations["sv-SE"].text)

2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:151 - analyze_text_syntax: language not supported by Google NLP: sv-SE
2026-04-19 15:12:14 - audio-language-trainer - INFO - nlp.py:189 - extract_lemmas_and_pos: falling back to spaCy for language 'sv-SE'
2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:197 - extract_lemmas_and_pos: neither Google NLP nor spaCy could process phrase for language 'sv-SE': Ett bättre alternativ
2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:151 - analyze_text_syntax: language not supported by Google NLP: sv-SE
2026-04-19 15:12:14 - audio-language-trainer - INFO - nlp.py:189 - extract_lemmas_and_pos: falling back to spaCy for language 'sv-SE'
2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:197 - extract_lemmas_and_pos: neither Google NLP nor spaCy could process phrase for language 'sv-SE': En cykel i låg växel
2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:151 - analyze_text_synt

{'verbs': [],
 'vocab': [],
 'tokens': ['och',
  'rent',
  'flicka',
  'person',
  'under',
  'för',
  'kort',
  'billig',
  'bättre',
  'färg',
  'pappersark',
  'ljus',
  'förändring',
  'Ett',
  'i',
  'cykel',
  'bänken',
  'genomtänkt',
  'flaska',
  'pojke',
  'varje',
  'glasring',
  'En',
  'växel',
  'svar',
  'alternativ',
  'låg',
  'en',
  'väster']}

In [ ]:
get_

In [ ]:
lm1000_tokens = get_unique_tokens_from_phrases(lm1000, "sv-SE")
lm1000_tokens = list(map(lambda x: x.lower(), lm1000_tokens))

In [ ]:
len(film_tokens)

In [ ]:
len(lm1000_tokens)

In [ ]:
(set(map(lambda x: x[:-1],film_tokens)).difference(set(lm1000_tokens)))

In [ ]:
film_tokens